# Ensemble Methods & Voting Classifier on Novagen Dataset

### What was learned & implemented in this notebook:
1. **Ensemble Modeling Techniques**:
   - Explored combining individual classification models (Logistic Regression, KNN, Random Forest, Gradient Boosting) into a single stronger learner.
2. **Gradient Boosting Classifier**:
   - Configured and trained `GradientBoostingClassifier` (n_estimators=500, learning_rate=0.1) achieving an impressive accuracy of **~94.3%** and recall of **~96.1%**.
3. **Voting Classifier (Soft Voting)**:
   - Configured `VotingClassifier` with Logistic Regression, KNN, and Random Forest estimators.
   - Employed `voting='soft'` (averaging predicted probabilities) to make final predictions, achieving an accuracy of **~91.5%**.

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,classification_report, recall_score
from sklearn.linear_model import LogisticRegression 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

In [2]:
df = pd.read_csv("novagen_dataset.csv")
df.sample(5)

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
3523,21.0,25.0,104.0,200.0,102.0,75.0,1.0,2.0,2.0,10.0,...,2,1,1,2,0,False,True,False,True,False
7747,11.0,26.0,138.0,199.0,101.0,76.0,9.0,1.0,3.0,5.0,...,2,0,1,0,1,False,False,False,True,False
8662,74.0,22.0,156.0,197.0,106.0,78.0,5.0,3.0,4.0,8.0,...,0,1,2,2,1,False,True,False,False,True
4710,48.0,26.0,163.0,201.0,100.0,75.0,8.0,2.0,1.0,4.0,...,2,2,0,1,2,False,False,False,True,False
7646,3.0,27.0,149.0,199.0,103.0,72.0,8.0,3.0,4.0,5.0,...,0,0,2,0,2,False,True,False,False,False


In [3]:
df.shape

(9549, 23)

In [4]:
df.describe()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,Target,Smoking,Alcohol,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies
count,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000,9549.000000
mean,33.806786,25.660697,130.382658,199.091528,100.225678,73.613782,6.951409,1.892345,3.580899,4.382134,0.521416,0.990470,0.995183,1.005864,0.998429,1.003351,1.004713,0.989318
std,24.566473,1.942369,27.878476,1.969234,2.157999,1.681538,2.352152,1.378714,1.622874,2.078593,0.499567,0.815521,0.816653,0.815877,0.821844,0.808800,0.813506,0.815699
min,0.000000,19.000000,22.000000,192.000000,93.000000,67.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,14.000000,24.000000,113.000000,198.000000,99.000000,73.000000,5.000000,1.000000,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,29.000000,26.000000,134.000000,199.000000,100.000000,74.000000,7.000000,2.000000,4.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,50.000000,27.000000,150.000000,200.000000,102.000000,75.000000,9.000000,3.000000,5.000000,6.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
max,100.000000,32.000000,225.000000,207.000000,107.000000,80.000000,14.000000,8.000000,10.000000,12.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

In [6]:
df.isnull().sum()

Age                      0
BMI                      0
Blood_Pressure           0
Cholesterol              0
Glucose_Level            0
Heart_Rate               0
Sleep_Hours              0
Exercise_Hours           0
Water_Intake             0
Stress_Level             0
Target                   0
Smoking                  0
Alcohol                  0
Diet                     0
MentalHealth             0
PhysicalActivity         0
MedicalHistory           0
Allergies                0
Diet_Type__Vegan         0
Diet_Type__Vegetarian    0
Blood_Group_AB           0
Blood_Group_B            0
Blood_Group_O            0
dtype: int64

In [7]:
X = df.drop("Target",axis =1)
y = df["Target"]
X_train, X_test,y_train , y_test = train_test_split(X,y, test_size = 0.2,random_state = 42, stratify = y)

X_test

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
5757,78.0,27.0,172.0,204.0,97.0,73.0,6.0,2.0,3.0,2.0,...,1,1,2,2,1,False,True,False,True,False
6047,2.0,24.0,151.0,198.0,100.0,71.0,11.0,2.0,4.0,1.0,...,2,2,2,0,2,False,True,True,False,False
3643,43.0,24.0,131.0,199.0,97.0,71.0,6.0,-0.0,3.0,3.0,...,2,2,0,2,1,True,False,False,False,True
150,43.0,29.0,124.0,198.0,96.0,72.0,6.0,3.0,4.0,3.0,...,1,2,2,1,0,False,True,True,False,False
4251,2.0,26.0,119.0,197.0,102.0,75.0,9.0,2.0,3.0,5.0,...,1,2,2,0,1,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8224,35.0,26.0,139.0,200.0,100.0,73.0,9.0,4.0,2.0,3.0,...,1,1,0,2,0,False,True,True,False,False
995,19.0,24.0,109.0,197.0,99.0,73.0,4.0,1.0,5.0,5.0,...,0,2,0,0,0,False,False,False,False,False
1137,24.0,27.0,136.0,199.0,101.0,74.0,8.0,2.0,4.0,4.0,...,2,1,2,0,1,True,False,False,False,True
8017,5.0,26.0,171.0,199.0,101.0,74.0,8.0,1.0,4.0,4.0,...,0,2,2,2,0,False,True,True,False,False


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
log_reg = LogisticRegression(
    penalty ="l2",
    solver = "liblinear",
    max_iter= 1000
)
log_reg.fit(X_train_scaled,y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8141361256544503
Logistic Regression Recall: 0.8283132530120482
              precision    recall  f1-score   support

           0       0.81      0.80      0.80       914
           1       0.82      0.83      0.82       996

    accuracy                           0.81      1910
   macro avg       0.81      0.81      0.81      1910
weighted avg       0.81      0.81      0.81      1910



In [10]:
knn = KNeighborsClassifier(
    n_neighbors = 5,
    metric = "euclidean"
)
knn.fit(X_train_scaled,y_train)
y_pred_knn = knn.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))


Logistic Regression Accuracy: 0.8832460732984293
Logistic Regression Recall: 0.8835341365461847
              precision    recall  f1-score   support

           0       0.87      0.88      0.88       914
           1       0.89      0.88      0.89       996

    accuracy                           0.88      1910
   macro avg       0.88      0.88      0.88      1910
weighted avg       0.88      0.88      0.88      1910



In [12]:
rf = RandomForestClassifier(
    n_estimators = 500,
    random_state = 42,
    max_depth = None
)
rf.fit(X_train_scaled,y_train)
y_pred_rf = rf.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))



Logistic Regression Accuracy: 0.9408376963350785
Logistic Regression Recall: 0.9628514056224899
              precision    recall  f1-score   support

           0       0.96      0.92      0.94       914
           1       0.93      0.96      0.94       996

    accuracy                           0.94      1910
   macro avg       0.94      0.94      0.94      1910
weighted avg       0.94      0.94      0.94      1910



In [16]:
gb = GradientBoostingClassifier(
    n_estimators = 500,
    learning_rate = 0.1,
    max_depth = 3,
    random_state =42
)
gb.fit(X_train_scaled,y_train)
y_pred_gb = gb.predict(X_test_scaled)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Gradient Boosting Recall:", recall_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))

Logistic Regression Accuracy: 0.9429319371727749
Logistic Regression Recall: 0.9608433734939759
              precision    recall  f1-score   support

           0       0.96      0.92      0.94       914
           1       0.93      0.96      0.95       996

    accuracy                           0.94      1910
   macro avg       0.94      0.94      0.94      1910
weighted avg       0.94      0.94      0.94      1910



In [19]:
vb = VotingClassifier(
    estimators = [
        ("lr",LogisticRegression(max_iter =1000,solver = "liblinear")),
        ("knn",KNeighborsClassifier(n_neighbors =5)),
        ("rf",RandomForestClassifier(n_estimators = 500,random_state= 42))
    ],
    voting = "soft"
)
vb.fit(X_train_scaled,y_train)
y_pred_vb = vb.predict(X_test_scaled)
print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred_vb))
print("Voting Classifier Recall:", recall_score(y_test, y_pred_vb))
print(classification_report(y_test, y_pred_vb))

Logistic Regression Accuracy: 0.9146596858638744
Logistic Regression Recall: 0.928714859437751
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       914
           1       0.91      0.93      0.92       996

    accuracy                           0.91      1910
   macro avg       0.92      0.91      0.91      1910
weighted avg       0.91      0.91      0.91      1910

